# 08. 안정성 패턴 — 장애에도 멈추지 않는 시스템 만들기

`난이도: 고급` · `소요: 35분` · `사전 지식:` [07. 고급 패턴](./07_advanced_patterns.ipynb) `권장`

---

## 목차
1. [Circuit Breaker](#Section-1:-Circuit-Breaker-패턴) — 집의 차단기처럼 장애 전파를 끊는다
2. [Cache-Aside](#Section-2:-Cache-Aside-패턴) — 도서관 사서처럼 캐시를 관리한다
3. [Retry with Backoff](#Section-3:-Retry-with-Backoff-데코레이터) — 전화 재다이얼처럼 점점 간격을 늘린다
4. [Kafka Transactional Producer](#Section-4:-Kafka-Transactional-Producer) — 은행 이체처럼 전부 성공하거나 전부 취소
5. [End-to-End Pipeline](#Section-5:-End-to-End-파이프라인) — 세 브로커를 하나로 연결
6. [Production Checklist](#Section-6:-Production-Checklist) — 운영 체크리스트

---

### 학습 목표
- Circuit Breaker의 3가지 상태 전환을 **집 차단기 비유**로 이해합니다
- Cache-Aside 패턴으로 캐시를 **자동으로 관리**하는 방법을 실습합니다
- Retry + Exponential Backoff가 **왜 필요한지** 체험합니다
- 세 브로커를 연결한 E2E 파이프라인을 직접 실행해봅니다

> **Tip** — 이 노트북의 패턴들은 실제 프로덕션에서 **필수적**입니다.
> 특히 Circuit Breaker와 Retry는 마이크로서비스 아키텍처의 기본 빌딩 블록입니다.

---

#### 노트북 네비게이션
| 이전 | 현재 | 다음 |
|------|------|------|
| [07. 고급 패턴](./07_advanced_patterns.ipynb) | **08. 안정성 패턴** | [09. 동시성 & 분산처리](./09_concurrency_and_distribution.ipynb) |

**전체 학습 경로**: `01 개요` → `02 Redis` · `03 RabbitMQ` · `04 Kafka` → `05 벤치마크` → `06 모니터링` → `07 고급패턴` → **`08 안정성`** → `09 동시성` → `10 Saga`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import asyncio
import json
import time
import httpx

print("08. 안정성 패턴 - Production-Grade Reliability")

---
## Section 1: Circuit Breaker 패턴

### 집의 전기 차단기(두꺼비집)와 같은 원리

집에서 전기를 너무 많이 쓰면 차단기가 내려가서 전체 정전을 막아줍니다.
Circuit Breaker 패턴도 같은 원리입니다.

```text
  [장애 없을 때 — CLOSED (차단기 올라감)]
  
  요청 → 서비스A → 정상 응답
  요청 → 서비스A → 정상 응답
  요청 → 서비스A → 정상 응답    ← 모든 요청이 통과됨
  
  ─────────────────────────────────────
  
  [장애 발생 — 서비스A가 죽었다면?]
  
  요청 → 서비스A → 타임아웃 (5초 대기)    ← 사용자가 5초 기다림
  요청 → 서비스A → 타임아웃 (5초 대기)    ← 또 5초...
  요청 → 서비스A → 타임아웃 (5초 대기)    ← 계속 5초씩 낭비!
  
  → 서비스A를 기다리는 동안 우리 서버의 스레드가 고갈
  → 우리 서비스도 함께 죽어버림! (장애 전파)
  
  ─────────────────────────────────────
  
  [Circuit Breaker가 있으면]
  
  요청 → 서비스A → 실패 (1회)
  요청 → 서비스A → 실패 (2회)
  요청 → 서비스A → 실패 (3회) → 임계치 도달!
  
  차단기 내려감! (OPEN)
  
  요청 → [즉시 거부] "서비스 일시 불가" (0.001초)  ← 기다리지 않음!
  요청 → [즉시 거부] "서비스 일시 불가" (0.001초)
  
  (10초 후... 서비스A가 복구되었을 수도 있으니 확인해보자)
  
  차단기 반쯤 올림 (HALF_OPEN)
  
  요청 1개만 → 서비스A → 성공!
  
  차단기 완전히 올림! (CLOSED) → 정상 운영 복귀
```

### 3가지 상태 전이 다이어그램

```text
                    실패 3번 누적
  ┌──────────┐  ────────────────→  ┌──────────┐
  │  CLOSED  │                     │   OPEN   │
  │ (정상)    │  ←────────────────  │ (차단)    │
  └──────────┘    테스트 성공       └──────────┘
       ↑                                │
       │          recovery_timeout 경과  │
       │                                ▼
       │                          ┌──────────┐
       └────── 테스트 성공 ───────│HALF_OPEN │
                                  │ (시험)    │
               테스트 실패 ──────→└──────────┘──→ 다시 OPEN
```

### 왜 그냥 재시도가 아니라 Circuit Breaker인가?

| | 재시도만 사용 | Circuit Breaker |
|---|-------------|-----------------|
| 장애 서비스에 요청 | 계속 보냄 (부하 악화) | 차단하여 보내지 않음 |
| 우리 서버 영향 | 타임아웃 대기로 리소스 고갈 | 즉시 거부 (리소스 보호) |
| 복구 확인 | 매번 시도해야 함 | HALF_OPEN에서 한 번만 확인 |

> **한 줄 요약** — "죽은 서비스에 계속 요청하지 말고, 차단기를 내려서 우리 서비스를 보호한다"

In [ ]:
from enum import Enum

class CircuitState(Enum):
    CLOSED = "closed"        # 정상 - 요청 통과
    OPEN = "open"            # 차단 - 즉시 실패
    HALF_OPEN = "half_open"  # 시험 - 일부 요청만 통과

print("Circuit Breaker 상태 정의 완료")
for s in CircuitState:
    print(f"  {s.name}: {s.value}")

### Circuit Breaker 핵심 구현

실패 횟수가 임계치(`failure_threshold`)를 넘으면 OPEN 상태로 전환하고,
`recovery_timeout` 이후에 HALF_OPEN으로 전환하여 복구를 시도합니다.

In [ ]:
class CircuitBreaker:
    def __init__(self, failure_threshold=3, recovery_timeout=10):
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.last_failure_time = None

    async def call(self, func, *args, **kwargs):
        if self.state == CircuitState.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN
            else:
                raise Exception("Circuit is OPEN - request blocked")

In [ ]:
    # call 메서드 나머지 부분 (성공/실패 처리)
        try:
            result = await func(*args, **kwargs)
            self._on_success()
            return result
        except Exception as e:
            self._on_failure()
            raise

    def _on_success(self):
        self.failure_count = 0
        self.state = CircuitState.CLOSED

    def _on_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN

# Note: 위 코드는 이전 셀의 클래스에 병합되어야 합니다.
# 아래에서 완전한 클래스를 재정의합니다.

In [ ]:
class CircuitBreaker:
    """완전한 Circuit Breaker 구현"""
    def __init__(self, failure_threshold=3, recovery_timeout=10):
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.last_failure_time = None
        self.call_log = []

    async def call(self, func, *args, **kwargs):
        if self.state == CircuitState.OPEN:
            elapsed = time.time() - self.last_failure_time
            if elapsed > self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN

In [ ]:
            else:
                self.call_log.append({"status": "blocked", "state": "OPEN"})
                raise Exception("Circuit is OPEN - request blocked")
        try:
            result = await func(*args, **kwargs)
            self._on_success()
            return result
        except Exception as e:
            self._on_failure()
            raise

    def _on_success(self):
        self.failure_count = 0
        self.state = CircuitState.CLOSED
        self.call_log.append({"status": "success", "state": "CLOSED"})

In [ ]:
    def _on_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
        self.call_log.append({
            "status": "failure",
            "state": self.state.value,
            "count": self.failure_count
        })

print("CircuitBreaker 클래스 정의 완료")

### Circuit Breaker 테스트

실패하는 함수를 반복 호출하여 상태 전이(CLOSED → OPEN → HALF_OPEN)를 확인합니다.

In [ ]:
call_count = 0

async def unreliable_service():
    """처음 4번은 실패, 이후 성공하는 서비스"""
    global call_count
    call_count += 1
    if call_count <= 4:
        raise ConnectionError(f"서비스 장애 (호출 #{call_count})")
    return {"status": "ok", "call": call_count}

print("테스트용 불안정 서비스 정의 완료")

In [ ]:
async def test_circuit_breaker():
    cb = CircuitBreaker(failure_threshold=3, recovery_timeout=2)
    for i in range(6):
        try:
            result = await cb.call(unreliable_service)
            print(f"  [{i}] 성공: {result} | 상태: {cb.state.value}")
        except Exception as e:
            print(f"  [{i}] 실패: {e} | 상태: {cb.state.value}")
        if cb.state == CircuitState.OPEN:
            print(f"  >> 복구 대기 2초...")
            await asyncio.sleep(2.5)
    return cb

cb = await test_circuit_breaker()
print(f"\n최종 상태: {cb.state.value}")

---
## Section 2: Cache-Aside 패턴

### 도서관 사서와 같은 원리

Cache-Aside(Lazy Loading)는 **도서관 사서**의 업무 방식과 같습니다.

```text
  [학생이 책을 요청할 때]
  
  학생: "자바의 정석 주세요"
  사서: (앞 책상 위에 있나 확인) ← Redis 캐시 확인
        ├── 있다! (Cache Hit) → 바로 전달 (0.1초)
        └── 없다  (Cache Miss)
              → 서고에서 찾아옴 (DB 조회, 5초)
              → 앞 책상 위에 한 권 올려놓음 (캐시에 저장)
              → 학생에게 전달
  
  다음 학생: "자바의 정석 주세요"
  사서: (앞 책상에 있네!) → 바로 전달 (0.1초)  ← 50배 빨라짐!
```

### Cache-Aside의 핵심 흐름

```text
  [읽기]                              [쓰기 (캐시 무효화)]
  
  1. Redis에서 찾기                    1. DB에 새 데이터 저장
     ├── Hit → 바로 반환               2. Redis에서 해당 키 삭제
     └── Miss                            → 다음 읽기 때 새 데이터가 캐시됨
         → 2. DB에서 읽기
         → 3. Redis에 저장 (TTL 설정)
         → 4. 반환
```

### 캐시 적중률(Hit Rate)이 중요한 이유

```text
  Hit Rate = 80% → DB 호출이 5분의 1로 줄어듦
  Hit Rate = 95% → DB 호출이 20분의 1로 줄어듦
  Hit Rate = 99% → DB 호출이 100분의 1로 줄어듦
  
  DB 서버 비용과 응답 속도에 직접적인 영향!
```

아래에서 Cache-Aside 패턴을 직접 구현하고, Hit Rate를 측정해봅시다.

In [ ]:
import redis.asyncio as aioredis

class CacheAside:
    def __init__(self, redis_client, default_ttl=300):
        self.redis = redis_client
        self.default_ttl = default_ttl
        self.stats = {"hits": 0, "misses": 0}

    async def get_or_fetch(self, key, fetch_func, ttl=None):
        cached = await self.redis.get(f"cache:{key}")
        if cached:
            self.stats["hits"] += 1
            return json.loads(cached)
        self.stats["misses"] += 1
        result = await fetch_func()

In [ ]:
        await self.redis.set(
            f"cache:{key}",
            json.dumps(result),
            ex=ttl or self.default_ttl
        )
        return result

    @property
    def hit_rate(self):
        total = self.stats["hits"] + self.stats["misses"]
        return self.stats["hits"] / total if total > 0 else 0

    async def invalidate(self, key):
        await self.redis.delete(f"cache:{key}")

print("CacheAside 클래스 정의 완료")

### Cache-Aside 테스트

시뮬레이션된 DB 호출로 캐시 적중률(Hit Rate)을 확인합니다.
첫 번째 호출은 Miss, 이후 동일 키 호출은 Hit이 됩니다.

In [ ]:
async def test_cache_aside():
    r = aioredis.from_url("redis://localhost:6379", decode_responses=True)
    cache = CacheAside(r, default_ttl=60)

    async def fake_db_query():
        await asyncio.sleep(0.1)  # DB 지연 시뮬레이션
        return {"user_id": 42, "name": "홍길동", "email": "hong@example.com"}

    # 첫 번째 호출 - Cache Miss
    t1 = time.time()
    result1 = await cache.get_or_fetch("user:42", fake_db_query)
    elapsed1 = time.time() - t1
    print(f"1차 호출 (Miss): {elapsed1:.3f}s | {result1}")

In [ ]:
    # 두 번째 호출 - Cache Hit
    t2 = time.time()
    result2 = await cache.get_or_fetch("user:42", fake_db_query)
    elapsed2 = time.time() - t2
    print(f"2차 호출 (Hit):  {elapsed2:.3f}s | {result2}")

    # 다른 키 호출 - Cache Miss
    async def fake_db_query_2():
        await asyncio.sleep(0.1)
        return {"user_id": 99, "name": "이순신"}

    await cache.get_or_fetch("user:99", fake_db_query_2)
    # 같은 키 반복 - Cache Hit
    await cache.get_or_fetch("user:42", fake_db_query)
    await cache.get_or_fetch("user:99", fake_db_query_2)

In [ ]:
    print(f"\n캐시 통계: {cache.stats}")
    print(f"적중률: {cache.hit_rate:.1%}")

    # 정리
    await cache.invalidate("user:42")
    await cache.invalidate("user:99")
    await r.aclose()
    return cache.stats

stats = await test_cache_aside()
print(f"\n최종 통계: {stats}")

---
## Section 3: Retry with Backoff 데코레이터

### 전화 재다이얼과 같은 원리

친구에게 전화를 걸었는데 안 받습니다. 어떻게 하시겠습니까?

```text
  [나쁜 방법 — 즉시 재다이얼]
  
  전화 → 안 받음 → 바로 다시 전화 → 안 받음 → 바로 다시 → 안 받음
  
  → 친구 핸드폰에 부재중 전화 100개 (스팸처럼!)
  → 통신사 네트워크도 과부하
  
  ──────────────────────────────────────
  
  [좋은 방법 — 점점 간격을 늘리며 재다이얼]
  
  전화 → 안 받음 → 1초 후 재시도
                 → 안 받음 → 2초 후 재시도
                           → 안 받음 → 4초 후 재시도
                                     → 받았다!
  
  → 적당한 간격을 두어 상대방이 전화받을 시간을 줌
  → 네트워크 부하도 줄어듦
```

### 왜 "지수(Exponential)" 백오프인가?

```text
  고정 간격:  1초 → 1초 → 1초 → 1초 (매번 같은 간격)
  지수 백오프: 1초 → 2초 → 4초 → 8초 → 16초 (2배씩 증가)
                                              ↑
                           서비스가 복구될 시간을 충분히 줌
```

### Jitter(무작위 변동)가 필요한 이유

서버가 장애를 일으키면, 모든 클라이언트가 **동시에 재시도**합니다.
이것을 **Thundering Herd(우르르 몰려가기)** 문제라고 합니다.

```text
  Jitter 없이:
  클라이언트 100개가 모두 1초 후 재시도 → 서버에 100개 동시 요청 → 또 장애!
  
  Jitter 있으면:
  클라이언트마다 0.5~1.5초 사이 랜덤 대기 → 분산된 재시도 → 서버 부하 분산
```

> **한 줄 요약** — "실패하면 점점 오래 기다렸다가 재시도하되, 타이밍을 랜덤하게 흩뿌린다"

In [ ]:
import functools
import random

def retry_with_backoff(max_retries=3, base_delay=1.0, max_delay=30.0, jitter=True):
    def decorator(func):
        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            for attempt in range(max_retries + 1):
                try:
                    return await func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries:
                        raise
                    delay = min(base_delay * (2 ** attempt), max_delay)
                    if jitter:

In [ ]:
                    delay *= random.uniform(0.5, 1.5)
                    print(f"  Retry {attempt+1}/{max_retries} after {delay:.1f}s: {e}")
                    await asyncio.sleep(delay)
        return wrapper
    return decorator

print("retry_with_backoff 데코레이터 정의 완료")

### Retry 데코레이터 테스트

3번째 시도에서 성공하는 함수로 백오프 동작을 확인합니다.
테스트 속도를 위해 `base_delay`를 짧게 설정합니다.

In [ ]:
retry_attempt = 0

@retry_with_backoff(max_retries=3, base_delay=0.1, jitter=False)
async def flaky_api_call():
    global retry_attempt
    retry_attempt += 1
    if retry_attempt < 3:
        raise ConnectionError(f"타임아웃 (시도 #{retry_attempt})")
    return {"data": "성공!", "attempt": retry_attempt}

print("재시도 데코레이터 테스트:")
result = await flaky_api_call()
print(f"최종 결과: {result}")

In [ ]:
# 모든 재시도가 실패하는 경우 테스트
@retry_with_backoff(max_retries=2, base_delay=0.05, jitter=False)
async def always_failing():
    raise RuntimeError("복구 불가능한 오류")

print("모든 재시도 실패 테스트:")
try:
    await always_failing()
except RuntimeError as e:
    print(f"최종 예외 발생: {e}")
    print(">> 재시도 횟수 소진 후 예외가 전파됩니다")

---
## Section 4: Kafka Transactional Producer

### 은행 계좌 이체와 같은 원리

A 계좌에서 B 계좌로 100만원을 이체할 때, 두 가지가 **반드시 함께** 일어나야 합니다:
1. A 계좌에서 100만원 출금
2. B 계좌에 100만원 입금

만약 1번만 성공하고 2번이 실패하면? **돈이 사라집니다!**

```text
  [트랜잭션 없이]
  토픽A에 "출금" 발행 → 성공
  토픽B에 "입금" 발행 → 실패!
  → 출금은 됐는데 입금은 안 됨 → 데이터 불일치!
  
  [트랜잭션 있으면]
  BEGIN TRANSACTION
    토픽A에 "출금" 발행 → 성공
    토픽B에 "입금" 발행 → 실패!
  ROLLBACK → 출금 발행도 취소됨 → 데이터 일관성 유지!
```

### 핵심 설정 3가지

| 설정 | 값 | 역할 |
|------|-----|------|
| `transactional_id` | `"txn-producer-1"` | 트랜잭션 식별자 (프로듀서 재시작 시에도 동일) |
| `enable_idempotence` | `True` | 네트워크 재시도로 인한 중복 메시지 방지 |
| `acks` | `"all"` | 모든 복제본에 기록 확인 후 ACK |

> **한 줄 요약** — "여러 토픽에 걸친 메시지를 은행 이체처럼 전부 성공하거나 전부 취소한다"

In [ ]:
from aiokafka import AIOKafkaProducer

async def kafka_transactional_demo():
    producer = AIOKafkaProducer(
        bootstrap_servers="localhost:9092",
        transactional_id="txn-producer-1",
        enable_idempotence=True,
        acks="all",
        value_serializer=lambda v: json.dumps(v).encode()
    )
    try:
        await producer.start()
        print("Kafka 트랜잭션 프로듀서 시작")
    except Exception as e:
        print(f"Kafka 연결 실패 (정상 - 로컬 환경): {e}")
        return None

In [ ]:
    try:
        async with producer.transaction():
            await producer.send("topic-a", {"event": "debit", "amount": 1000})
            await producer.send("topic-b", {"event": "credit", "amount": 1000})
            print("트랜잭션 커밋: debit + credit 모두 발행")
    except Exception as e:
        print(f"트랜잭션 실패 (롤백됨): {e}")
    finally:
        await producer.stop()
        print("프로듀서 종료")

await kafka_transactional_demo()

### Kafka Idempotent Producer (트랜잭션 없이)

트랜잭션까지 필요하지 않은 경우에도 **멱등성(Idempotence)**을 활성화하면
네트워크 재시도로 인한 **중복 메시지를 자동으로 방지**할 수 있습니다.

In [ ]:
async def kafka_idempotent_demo():
    producer = AIOKafkaProducer(
        bootstrap_servers="localhost:9092",
        enable_idempotence=True,
        acks="all",
        value_serializer=lambda v: json.dumps(v).encode()
    )
    try:
        await producer.start()
        for i in range(3):
            await producer.send("events", {"seq": i, "type": "order"})
        print("멱등 프로듀서: 3개 메시지 발행 (중복 방지)")
    except Exception as e:
        print(f"Kafka 연결 실패 (정상): {e}")
    finally:
        await producer.stop()

await kafka_idempotent_demo()

---
## Section 5: End-to-End 파이프라인

### 전체 흐름: API → Kafka 발행 → 처리 → Redis 캐시 → RabbitMQ 알림

실제 프로덕션 환경에서는 여러 패턴을 **조합**하여 사용합니다.
아래는 주문 처리 파이프라인의 각 단계를 시뮬레이션합니다:

1. FastAPI 엔드포인트로 주문 요청
2. Kafka로 이벤트 발행 (with Circuit Breaker)
3. 처리 결과를 Redis에 캐시
4. RabbitMQ로 알림 발송

In [ ]:
async def simulate_api_call(order_id: int):
    """FastAPI 엔드포인트 호출 시뮬레이션"""
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(
                f"http://localhost:8000/health",
                timeout=5.0
            )
            print(f"  [API] 응답: {resp.status_code}")
            return resp.json()
    except Exception as e:
        print(f"  [API] 연결 실패 (시뮬레이션 모드): {e}")
        return {"order_id": order_id, "status": "simulated"}

In [ ]:
async def simulate_kafka_publish(event: dict):
    """Kafka 발행 시뮬레이션 (Circuit Breaker 적용)"""
    print(f"  [Kafka] 이벤트 발행: {event['type']}")
    # 실제 환경에서는 AIOKafkaProducer.send() 호출
    await asyncio.sleep(0.05)
    return {"topic": "orders", "offset": 42}

async def simulate_redis_cache(key: str, data: dict):
    """Redis 캐시 저장 시뮬레이션"""
    print(f"  [Redis] 캐시 저장: {key}")
    await asyncio.sleep(0.01)
    return True

In [ ]:
async def simulate_rabbitmq_notify(message: str):
    """RabbitMQ 알림 발송 시뮬레이션"""
    print(f"  [RabbitMQ] 알림: {message}")
    await asyncio.sleep(0.02)
    return True

print("파이프라인 구성 요소 정의 완료")

### 파이프라인 실행

Circuit Breaker와 Retry를 조합하여 전체 파이프라인을 안정적으로 실행합니다.

In [ ]:
async def run_order_pipeline(order_id: int):
    """주문 처리 E2E 파이프라인"""
    cb = CircuitBreaker(failure_threshold=3, recovery_timeout=5)
    print(f"=== 주문 #{order_id} 처리 시작 ===")

    # Step 1: API 호출
    api_result = await simulate_api_call(order_id)

    # Step 2: Kafka 발행 (Circuit Breaker)
    event = {"type": "order_created", "order_id": order_id}
    kafka_result = await cb.call(simulate_kafka_publish, event)

    # Step 3: Redis 캐시
    await simulate_redis_cache(f"order:{order_id}", api_result)

In [ ]:
    # Step 4: RabbitMQ 알림
    await simulate_rabbitmq_notify(f"주문 #{order_id} 처리 완료")

    print(f"=== 주문 #{order_id} 처리 완료 ===")
    return {"order_id": order_id, "status": "completed"}

# 3개 주문 병렬 처리
results = await asyncio.gather(
    run_order_pipeline(1001),
    run_order_pipeline(1002),
    run_order_pipeline(1003)
)
print(f"\n처리 결과: {len(results)}건 완료")

### httpx를 사용한 실제 API 엔드포인트 호출

FastAPI 서버가 실행 중이라면, 실제 엔드포인트를 호출하여 전체 흐름을 확인할 수 있습니다.

In [ ]:
async def call_api_endpoints():
    """FastAPI 엔드포인트 호출 (서버 실행 시)"""
    base_url = "http://localhost:8000"
    endpoints = ["/health", "/docs"]

    async with httpx.AsyncClient(timeout=3.0) as client:
        for ep in endpoints:
            try:
                resp = await client.get(f"{base_url}{ep}")
                print(f"  {ep}: {resp.status_code}")
            except httpx.ConnectError:
                print(f"  {ep}: 서버 미실행 (정상)")

await call_api_endpoints()

---
## Section 6: Production Checklist

### 운영 환경 배포 전 체크리스트

각 브로커별로 반드시 확인해야 할 항목들입니다.

| 구분 | 항목 | Kafka | RabbitMQ | Redis |
|------|------|-------|----------|-------|
| **연결** | Connection Pooling | `max_batch_size` 설정 | `connection_pool_size` | `ConnectionPool` 사용 |
| **연결** | 재연결 로직 | `retry_backoff_ms` | `heartbeat=600` | `retry_on_timeout=True` |
| **에러** | Circuit Breaker | 프로듀서/컨슈머 분리 | Exchange 별 분리 | 읽기/쓰기 분리 |
| **에러** | Dead Letter Queue | DLT 토픽 설정 | DLX Exchange 설정 | 별도 키 저장 |
| **에러** | Retry 전략 | `retries`, `retry_backoff_ms` | `x-retry-count` 헤더 | `RETRY` 커맨드 없음 |
| **성능** | Batch 처리 | `batch_size`, `linger_ms` | `prefetch_count` | Pipeline 사용 |
| **모니터링** | 메트릭 수집 | JMX / Prometheus | Management Plugin | `INFO` 커맨드 |
| **모니터링** | 로깅 | 구조화 로깅 필수 | 구조화 로깅 필수 | Slow Log 활성화 |
| **보안** | 인증 | SASL/SSL | TLS + 사용자 인증 | `requirepass` + TLS |
| **데이터** | 직렬화 | Avro/Protobuf 권장 | JSON/MessagePack | JSON/MessagePack |

### 패턴 선택 가이드

| 상황 | 권장 패턴 | 이유 |
|------|-----------|------|
| 외부 API 호출 실패 반복 | **Circuit Breaker** | 장애 서비스에 계속 요청하면 자원 낭비 |
| DB 조회 응답 느림 | **Cache-Aside** | Redis로 응답 시간 10x 이상 개선 가능 |
| 일시적 네트워크 오류 | **Retry with Backoff** | 잠시 후 재시도하면 대부분 성공 |
| 여러 토픽에 원자적 발행 | **Transactional Producer** | 부분 실패 시 데이터 정합성 문제 |
| 메시지 중복 수신 | **Idempotent Consumer** | 같은 메시지를 여러 번 처리해도 결과 동일 |

### 핵심 원칙 요약

1. **Fail Fast**: 장애가 감지되면 즉시 실패 (Circuit Breaker)
2. **Graceful Degradation**: 일부 기능이 실패해도 서비스는 계속 동작
3. **Idempotency**: 같은 요청을 여러 번 보내도 결과가 동일
4. **Observability**: 모든 패턴에 로깅과 메트릭을 반드시 추가
5. **Defense in Depth**: 단일 패턴에 의존하지 말고 여러 패턴을 조합

In [ ]:
# 리소스 정리
print("=== 리소스 정리 ===")
print("모든 시뮬레이션 리소스가 정리되었습니다.")
print()
print("학습한 패턴:")
print("  1. Circuit Breaker - 장애 전파 방지")
print("  2. Cache-Aside - Redis 캐시 계층")
print("  3. Retry with Backoff - 지수 백오프 재시도")
print("  4. Kafka Transactional Producer - 원자적 발행")
print("  5. E2E Pipeline - 전체 흐름 통합")
print("\n08. 안정성 패턴 노트북 완료!")